# Lab 1: Làm quen với PyTorch

PyTorch là một thư viện deep learning rất phổ biến, được phát triển bởi Facebook AI Research. Nó được thiết kế để vừa mạnh mẽ (chạy GPU), vừa Pythonic (gần với NumPy) — nên rất dễ học cho ai đã quen NumPy.

Ở bài này ta sẽ đi qua những thứ tối thiểu cần biết trước khi bắt tay vào các mô hình thật:
1. **Tensor** — "viên gạch" cơ bản, giống ndarray của NumPy nhưng chạy được trên GPU.
2. **GPU và device** — kiểm tra GPU có sẵn không, chuyển dữ liệu giữa CPU/GPU.
3. **Autograd** — cơ chế tính đạo hàm tự động, là "phép thuật" giúp ta train được mạng nơ-ron.
4. **Một ví dụ Gradient Descent thủ công** — ghép các thứ trên lại để hiểu cách model thực sự học.
5. **Bài tập** ở cuối để các em luyện tay.

Mở Colab hoặc Jupyter, chạy từng cell theo thứ tự.

## 1. Kiểm tra GPU

Trước khi train một mạng to, mình thường muốn biết: máy này có GPU không, và nếu có thì là GPU gì.

Lưu ý: rất nhiều bạn lần đầu hay viết `torch.cuda.current_device` — quên dấu ngoặc cuối — kết quả ra chữ "function ..." thay vì số id của thiết bị. Phải có dấu `()` mới gọi được hàm.

In [ ]:
import torch

print('GPU có sẵn không?', torch.cuda.is_available())
print('Phiên bản PyTorch:', torch.__version__)

if torch.cuda.is_available():
    print('Số GPU:', torch.cuda.device_count())
    print('Device đang dùng:', torch.cuda.current_device())   # nhớ có ()
    print('Tên GPU:', torch.cuda.get_device_name(0))
    print(f'Bộ nhớ đã cấp phát: {torch.cuda.memory_allocated() / 1e6:.2f} MB')
    print(f'Bộ nhớ được giữ:    {torch.cuda.memory_reserved()  / 1e6:.2f} MB')
    # Lưu ý: hàm `memory_cached` đã bị deprecated từ PyTorch 1.6, dùng `memory_reserved` thay thế.
else:
    print('Không có GPU — sẽ chạy trên CPU. Vẫn ổn cho các bài lab nhỏ.')

# Khai báo device để dùng xuyên suốt notebook.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('\nDevice sẽ dùng:', device)

## 2. Tensor — "viên gạch" của PyTorch

Tensor là cách PyTorch lưu mọi thứ: ảnh, vector, ma trận, batch dữ liệu... Nó tương tự `numpy.ndarray` nhưng có thêm 2 đặc tính quan trọng:
1. Có thể chạy trên GPU (`.cuda()` hoặc `.to(device)`).
2. Có thể tính gradient tự động (nếu `requires_grad=True`).

### 2.1. Tạo tensor

In [ ]:
import numpy as np
torch.manual_seed(42)

# Một số cách phổ biến để tạo tensor.
a = torch.tensor([1, 2, 3])                  # từ list Python
b = torch.zeros(2, 3)                        # ma trận 2×3 toàn 0
c = torch.ones(3)                            # vector 3 phần tử toàn 1
d = torch.empty(2, 2)                        # cấp phát chỗ nhưng chưa khởi tạo (giá trị rác)
e = torch.randn(2, 3)                        # phân phối chuẩn N(0, 1)
f = torch.arange(0, 10, 2)                   # 0, 2, 4, 6, 8
g = torch.linspace(0, 1, 5)                  # 5 số đều nhau từ 0 đến 1

print('a =', a)
print('b =\n', b)
print('c =', c)
print('e =\n', e)
print('f =', f)
print('g =', g)

# In các thuộc tính quan trọng.
print('\nthuộc tính của e:')
print('  shape:', e.shape)
print('  dtype:', e.dtype)
print('  device:', e.device)

### 2.2. Chuyển qua lại với NumPy — coi chừng cùng bộ nhớ

Hai cách chuyển ndarray → tensor có hành vi *khác nhau*:

- **`torch.from_numpy(arr)`** — chia sẻ bộ nhớ với arr. Sửa arr → tensor cũng đổi theo.
- **`torch.tensor(arr)`** — sao chép dữ liệu sang vùng nhớ mới. Sửa arr → tensor không đổi.

Thử nghiệm cụ thể luôn để khỏi nhầm.

In [ ]:
arr = np.arange(0, 5)
t1 = torch.from_numpy(arr)   # chia sẻ bộ nhớ
t2 = torch.tensor(arr)       # copy

print('Trước khi sửa arr:')
print('  arr =', arr)
print('  t1  =', t1)
print('  t2  =', t2)

arr[0] = 99   # đổi giá trị đầu tiên của arr
print('\nSau khi sửa arr[0] = 99:')
print('  arr =', arr)
print('  t1  =', t1, '  ← thay đổi vì from_numpy chia sẻ bộ nhớ')
print('  t2  =', t2, '  ← không đổi vì tensor() đã copy')

### 2.3. Reshape — `view` và `reshape`

Hai hàm này gần như tương đương. Khác biệt nhỏ: `view` yêu cầu tensor phải *contiguous* trong bộ nhớ; `reshape` thì tự xử lý nếu cần copy. Mới học cứ dùng `reshape` cho an toàn.

In [ ]:
x = torch.arange(12)
print('Original:', x.shape)

y1 = x.view(3, 4)
y2 = x.reshape(3, 4)
y3 = x.view(-1, 4)   # -1 là "tự tính cho khớp"

print('view(3,4):\n', y1)
print('view(-1,4):\n', y3)

### 2.4. Chuyển tensor lên GPU

Khi train mạng, model và dữ liệu phải cùng nằm trên cùng một device. Quy tắc: dùng `.to(device)` cho mọi thứ.

In [ ]:
x = torch.randn(3, 3)
print('Trên CPU:', x.device)

x = x.to(device)
print('Sau .to(device):', x.device)

# Một sai lầm phổ biến: tensor ở CPU, model ở GPU → lỗi runtime.
# Kinh nghiệm: ngay khi load 1 batch dữ liệu, .to(device) ngay lập tức.

## 3. Autograd — đạo hàm tự động

Đây là phần "thần kỳ" của PyTorch: nó tự ghi lại các phép toán mình làm trên tensor có `requires_grad=True`, sau đó khi gọi `.backward()` thì tự tính đạo hàm theo chain rule.

### 3.1. Ví dụ một biến: tính đạo hàm $y = 2x^4 + x^3 + 3x^2 + 5x + 1$ tại $x = 2$

Bằng tay: $y' = 8x^3 + 3x^2 + 6x + 5$. Tại $x = 2$: $y' = 64 + 12 + 12 + 5 = 93$. Xem PyTorch có cho ra cùng kết quả không.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)   # bật autograd cho biến này
y = 2*x**4 + x**3 + 3*x**2 + 5*x + 1
print(f'y = {y.item()}')

y.backward()              # tính đạo hàm dy/dx và lưu vào x.grad
print(f'dy/dx tại x=2: {x.grad.item()}')
# Đúng bằng 93 như tính tay.

### 3.2. Ví dụ nhiều biến

Cho $f(a, b) = a^2 + 3ab + b^3$. Tính $\partial f/\partial a$ và $\partial f/\partial b$ tại $a = 2, b = 3$.

Bằng tay:
$$\frac{\partial f}{\partial a} = 2a + 3b = 4 + 9 = 13$$
$$\frac{\partial f}{\partial b} = 3a + 3b^2 = 6 + 27 = 33$$

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)
f = a**2 + 3*a*b + b**3
f.backward()
print(f'df/da = {a.grad.item()}   (mong đợi 13)')
print(f'df/db = {b.grad.item()}   (mong đợi 33)')

### 3.3. Một quy tắc quan trọng: gradient *cộng dồn*

Mỗi lần `.backward()`, gradient được **cộng vào** `x.grad` chứ không ghi đè. Đó là lý do trong vòng lặp train ta phải gọi `optimizer.zero_grad()` trước mỗi bước — để xoá gradient cũ.

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
for i in range(3):
    y = x ** 2
    y.backward()
    print(f'Lần {i+1}: x.grad = {x.grad.item()}  (gradient bị cộng dồn!)')

print('\nGiờ thử zero_grad sau mỗi bước:')
x = torch.tensor(1.0, requires_grad=True)
for i in range(3):
    if x.grad is not None:
        x.grad.zero_()
    y = x ** 2
    y.backward()
    print(f'Lần {i+1}: x.grad = {x.grad.item()}')

## 4. Ví dụ Gradient Descent thủ công

Giờ ráp tất cả lại để tìm cực tiểu của $y = (x - 3)^2 + 1$ bằng gradient descent thủ công. Cực tiểu nằm tại $x = 3$, $y = 1$.

Quy tắc cập nhật: $x \leftarrow x - \alpha \cdot \partial y / \partial x$ với $\alpha$ là learning rate.

Ta khởi đầu từ $x = 0$, dùng $\alpha = 0.1$, lặp 30 vòng.

In [ ]:
import matplotlib.pyplot as plt

x = torch.tensor(0.0, requires_grad=True)
lr = 0.1
history = []

for step in range(30):
    y = (x - 3)**2 + 1
    y.backward()

    # Cập nhật x bằng gradient descent.
    # `with torch.no_grad()` để tắt việc track autograd cho phép cập nhật này
    # — vì cập nhật trọng số không cần được đưa vào graph tính đạo hàm.
    with torch.no_grad():
        x -= lr * x.grad
        x.grad.zero_()           # phải zero_grad sau khi cập nhật

    history.append((x.item(), y.item()))
    if step % 5 == 0 or step == 29:
        print(f'step {step:2d}: x = {x.item():.4f}, y = {y.item():.4f}')

xs, ys = zip(*history)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(xs, 'o-'); axes[0].axhline(3, color='red', linestyle='--', label='cực tiểu')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('x'); axes[0].legend(); axes[0].set_title('x hội tụ về 3')
axes[1].plot(ys, 'o-'); axes[1].set_xlabel('Step'); axes[1].set_ylabel('y'); axes[1].set_title('y giảm dần về 1')
plt.tight_layout(); plt.show()

Quan sát: x từ 0 dần tiến về 3, y từ 10 giảm về xấp xỉ 1. Đó chính là *gradient descent đang hoạt động* — và đây cũng chính là cơ chế bên dưới mọi `optimizer.step()` mà em sẽ thấy ở các bài sau.

### Nhận xét về learning rate
Thử đổi `lr = 1.0` và chạy lại — em sẽ thấy x bị nhảy quanh, không hội tụ. Thử `lr = 0.01` — x hội tụ nhưng quá chậm. Chọn lr đúng là một nghệ thuật, không có công thức cố định.

# BÀI TẬP VỀ NHÀ

## Bài 1: Đạo hàm và Gradient Descent
Cho hàm $y = x^3 + 2x^2 + 5x + 1$.

1. Tạo tensor `x = 2.0` với `requires_grad=True`. Tính $dy/dx$ tại $x = 2$. So sánh với kết quả tính tay (đáp án bằng tay là $3 \cdot 4 + 4 \cdot 2 + 5 = 25$).
2. Áp dụng gradient descent với learning rate $\alpha = 0.01$ (KHÔNG dùng 0.1 — sẽ phân kỳ vì gradient quá lớn ở vị trí ban đầu) trong 50 vòng lặp. Vẽ giá trị $x$ và $y$ qua từng vòng.
3. Nhận xét: x hội tụ về đâu? Có phải cực tiểu thật của hàm không?

*Gợi ý:* hàm này có cực tiểu tại $x \approx -0.74$ (tìm bằng cách giải $3x^2 + 4x + 5 = 0$ → vô nghiệm thực, nên hàm đơn điệu tăng → x đi về $-\infty$). Đây là một bài học hay: **GD trên hàm không có cực tiểu sẽ chạy mãi không dừng**.

## Bài 2: Hồi quy tuyến tính bằng GD thủ công
Tạo dữ liệu giả lập:
- $x$: 50 điểm ngẫu nhiên trong $[1, 10]$.
- $y = 3x + 5 + \varepsilon$ với $\varepsilon \sim N(0, 0.5)$.

Yêu cầu:
1. Khởi tạo `w = 0.0`, `b = 0.0` với `requires_grad=True`.
2. Định nghĩa loss MSE: $L = \frac{1}{N}\sum (y_i - (wx_i + b))^2$.
3. Train trong 200 vòng với `lr = 0.01`. Mỗi 20 vòng in ra `w`, `b`, `loss`.
4. Vẽ đường thẳng học được kèm dữ liệu.
5. So sánh `w, b` cuối cùng với giá trị thật `(3, 5)`.

*Gợi ý:* nhớ `optimizer.zero_grad()` (hoặc `w.grad.zero_(); b.grad.zero_()`) sau mỗi vòng.

## Bài 3: Tạo tensor với các phương thức khác nhau
Viết một cell tạo và in ra:
- `torch.empty(3, 4)`
- `torch.zeros(2, 3)`
- `torch.ones(4)`
- `torch.rand(3, 3)` — phân phối đều trong [0, 1]
- `torch.randn(3, 3)` — phân phối chuẩn N(0, 1)
- Tạo một tensor `t` có shape `(2, 6)` rồi dùng `view` và `view_as` để biến nó thành shape `(3, 4)` (gợi ý: `view_as` cần một tensor mẫu có shape mong muốn).

## Bài 4: Hàm hai biến
Cho $f(x, y) = x^2 + 2y^2 + 3xy + x + y$. Tại $(x, y) = (1, 2)$:
1. Tính $\partial f / \partial x$ và $\partial f / \partial y$ bằng PyTorch.
2. Kiểm tra bằng cách tính tay (đáp án: $\partial f / \partial x = 2x + 3y + 1 = 9$, $\partial f / \partial y = 4y + 3x + 1 = 12$).
3. Áp dụng gradient descent đồng thời cho cả $x$ và $y$, lr = 0.05, 30 vòng. Vẽ quỹ đạo $(x, y)$ trên mặt phẳng.

## Deadline
16/03/2026.